# Comparación de modelos sobre `test.jsonl`

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "evaluation").exists() else Path.cwd().parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evaluation.run_baseline import (
    aggregate_by,
    calibration,
    k_values_for,
    load_tasks,
    load_traces,
    summarize_model,
)

LEVELS = ["easy", "medium", "hard", "expert"]
K = 8
BASE_KEY = "Base (Ollama q4)"

# Acepta reportes .json (run_baseline / run_hf) o trazas .jsonl (completas o parciales).
MODELS = {
    "Base (Ollama q4)": "outputs/baseline_test/baseline.json",
    "SFT (LoRA)":       "outputs/sft_lora_test/sft_lora.json",
    "DPO (LoRA)":       "outputs/dpo_lora_test/traces_dpo_lora.jsonl",
    # Cuando termine DPO o GRPO, actualiza o descomenta la ruta:
    # "DPO (LoRA)":       "outputs/dpo_lora_test/dpo_lora.json",
    # "GRPO binaria":     "outputs/grpo_binaria_test/grpo_binaria.json",
    # "GRPO relajada":    "outputs/grpo_relajada_test/grpo_relajada.json",
    # "GRPO estricta":    "outputs/grpo_estricta_test/grpo_estricta.json",
}

In [ ]:
tasks = load_tasks("test")

def load_summary(path_str):
    p = Path(path_str)
    if not p.exists():
        print(f"[omitido] No existe: {p}")
        return None
    if p.suffix == ".json":
        data = json.loads(p.read_text(encoding="utf-8"))
        return next(iter(data["models"].values()))
    done = load_traces(p)
    complete = [t for t in tasks if all((t.id, s) in done for s in range(K))]
    episodes = [[done[(t.id, s)] for s in range(K)] for t in complete]
    return summarize_model(complete, episodes, K)

def add_delta_cols(df, metric_cols, base_name=BASE_KEY):
    """Inserta una columna Δ (%) vs Base al lado de cada métrica ('-' para Base)."""
    if base_name not in df["Modelo"].values:
        return df
    base = df.loc[df["Modelo"] == base_name].iloc[0]
    out = df[[c for c in df.columns if c not in metric_cols]].copy()
    for c in metric_cols:
        out[c] = df[c]
        deltas = []
        for _, row in df.iterrows():
            if row["Modelo"] == base_name or pd.isna(base[c]) or pd.isna(row[c]) or base[c] == 0:
                deltas.append("-")
            else:
                pct = ((row[c] - base[c]) / base[c]) * 100
                deltas.append(f"{pct:+.1f}%")
        out[f"Δ {c}"] = deltas
    return out

def build_subset_tables(selected_names):
    common_ids = set.intersection(*(
        {r["task_id"] for r in summaries[n]["per_task"]} for n in selected_names
    ))
    main_rows, diff_rows, block_rows, cal_rows = [], [], [], []
    by_diff_map, by_block_map = {}, {}
    for name in selected_names:
        sub = [r for r in summaries[name]["per_task"] if r["task_id"] in common_ids]
        ks = k_values_for(min(r["n"] for r in sub), K)
        by_d = aggregate_by(sub, "difficulty", ks)
        by_b = aggregate_by(sub, "block", ks)
        cal = calibration(sub)["all"]
        by_diff_map[name] = by_d
        by_block_map[name] = by_b
        main_rows.append({
            "Modelo": name, "Tareas": len(sub),
            "pass@1": by_d["all"].get("pass@1"), "pass@2": by_d["all"].get("pass@2"),
            "pass@4": by_d["all"].get("pass@4"), "pass@8": by_d["all"].get("pass@8"),
        })
        block_rows.append({
            "Modelo": name, "Tareas": len(sub),
            "seen@1": by_b.get("seen", {}).get("pass@1"),
            "heldout@1": by_b.get("heldout", {}).get("pass@1"),
            "seen@8": by_b.get("seen", {}).get("pass@8"),
            "heldout@8": by_b.get("heldout", {}).get("pass@8"),
        })
        cal_rows.append({
            "Modelo": name, "Tareas": len(sub),
            "Nunca": cal["never_solved"], "Zona": cal["learnable"], "Siempre": cal["always_solved"],
        })
        diff_rows.append({
            "Modelo": name, "Tareas": len(sub),
            **{lvl: by_d.get(lvl, {}).get("pass@1") for lvl in LEVELS + ["all"]},
        })
    return {
        "n": len(common_ids),
        "passk": add_delta_cols(pd.DataFrame(main_rows), ["pass@1", "pass@2", "pass@4", "pass@8"]),
        "block": add_delta_cols(pd.DataFrame(block_rows), ["seen@1", "heldout@1", "seen@8", "heldout@8"]),
        "calib": add_delta_cols(pd.DataFrame(cal_rows), ["Nunca", "Zona", "Siempre"]),
        "diff":  add_delta_cols(pd.DataFrame(diff_rows), LEVELS + ["all"]),
        "by_diff": by_diff_map,
        "by_block": by_block_map,
    }

summaries = {}
for name, path in MODELS.items():
    s = load_summary(path)
    if s and s.get("n_tasks", 0) > 0:
        summaries[name] = s
        tag = "completo" if s["n_tasks"] == len(tasks) else f"parcial {s['n_tasks']}/{len(tasks)}"
        print(f"{name:<20} -> {s['n_tasks']} tareas x k={s['samples_per_task']} ({tag})")

## 1. Modelos completos (602 tareas de `test.jsonl`)

In [ ]:
full_names = [n for n, s in summaries.items() if s["n_tasks"] == len(tasks)]
full = build_subset_tables(full_names)
print(f"Modelos completos ({full['n']} tareas): {full_names}")
display(full["passk"])
display(full["block"])
display(full["calib"])
display(full["diff"])

## 2. Todos los modelos sobre tareas en común (incluye corridas parciales como DPO)

In [ ]:
all_names = list(summaries.keys())
common = build_subset_tables(all_names)
print(f"Comparación justa en común ({common['n']} tareas): {all_names}")
display(common["passk"])
display(common["block"])
display(common["calib"])
display(common["diff"])

## 3. Gráficos comparativos (`pass@1` por dificultad y `seen` vs `heldout`)

In [ ]:
names = all_names
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x = np.arange(len(LEVELS))
w = 0.8 / max(1, len(names))
for i, name in enumerate(names):
    vals = [common["by_diff"][name].get(lvl, {}).get("pass@1", 0) for lvl in LEVELS]
    axes[0].bar(x + (i - (len(names) - 1) / 2) * w, vals, width=w, label=name)
axes[0].set_xticks(x)
axes[0].set_xticklabels(LEVELS)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("pass@1")
axes[0].set_title(f"pass@1 por dificultad (n={common['n']})")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

blocks = ["seen", "heldout"]
xb = np.arange(len(blocks))
for i, name in enumerate(names):
    vals = [common["by_block"][name].get(b, {}).get("pass@1", 0) for b in blocks]
    axes[1].bar(xb + (i - (len(names) - 1) / 2) * w, vals, width=w, label=name)
axes[1].set_xticks(xb)
axes[1].set_xticklabels(blocks)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("pass@1")
axes[1].set_title(f"Generalización seen vs heldout (n={common['n']})")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Etapas intermedias y uso de herramienta

In [ ]:
stage_rows, tool_rows = [], []
for name, s in summaries.items():
    tu = s["tool_usage"]
    tool_rows.append({
        "Modelo": name, "Tareas": s["n_tasks"],
        "Formato OK": tu["valid_format_rate"],
        "SQL emitido": tu["sql_emitted_rate"],
        "SQL ejecutable": tu["executable_sql_rate"],
        "Turnos prom": tu["avg_turns"],
    })
    sr = s.get("stage_rates", {}).get("all")
    if sr:
        stage_rows.append({
            "Modelo": name,
            "Episodios": sr["n_episodes"],
            "SQL ejecutable": sr["sql_executable"],
            "Resultado = GT": sr["result_correct"],
            "Respuesta correcta": sr["answer_correct"],
            "Brecha SQL->GT": round(sr["sql_executable"] - sr["result_correct"], 4),
            "Brecha GT->Ans": round(sr["result_correct"] - sr["answer_correct"], 4),
        })

df_tool = add_delta_cols(pd.DataFrame(tool_rows), ["Formato OK", "SQL emitido", "SQL ejecutable", "Turnos prom"])
df_stages = pd.DataFrame(stage_rows)
display(df_tool)
display(df_stages) if not df_stages.empty else print("Sin stage_rates aún.")

## 5. Exportar todas las tablas a Excel (`.xlsx`) para copiar directo a Docs

In [ ]:
out_xlsx = Path("outputs/comparacion_modelos.xlsx")
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    full["passk"].to_excel(writer, sheet_name=f"passk_{full['n']}", index=False)
    full["block"].to_excel(writer, sheet_name=f"seen_heldout_{full['n']}", index=False)
    full["calib"].to_excel(writer, sheet_name=f"calibracion_{full['n']}", index=False)
    full["diff"].to_excel(writer, sheet_name=f"dificultad_{full['n']}", index=False)
    if common["n"] != full["n"]:
        common["passk"].to_excel(writer, sheet_name=f"passk_comun_{common['n']}", index=False)
        common["block"].to_excel(writer, sheet_name=f"seen_heldout_comun_{common['n']}", index=False)
        common["calib"].to_excel(writer, sheet_name=f"calibracion_comun_{common['n']}", index=False)
        common["diff"].to_excel(writer, sheet_name=f"dificultad_comun_{common['n']}", index=False)
    df_tool.to_excel(writer, sheet_name="uso_herramienta", index=False)
    if not df_stages.empty:
        df_stages.to_excel(writer, sheet_name="etapas_intermedias", index=False)

common["passk"].to_csv("outputs/comparacion_modelos.csv", index=False)
print(f"Excel listo en: {out_xlsx.resolve()}")